In [ ]:
# 1. Mount Google Drive to save the model later safely
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
import os

# 1. Download standard Houses Dataset (Images + Text) directly from GitHub
!git clone https://github.com/emanhamed/Houses-dataset.git

# 2. Load the Tabular Data (Text details)
txt_path = "/content/Houses-dataset/Houses Dataset/HousesInfo.txt"
cols = ["Bedrooms", "Bathrooms", "Area", "ZipCode", "Price"]
df = pd.read_csv(txt_path, sep=" ", header=None, names=cols)

# 3. Verify the data
print(" Dataset Downloaded Successfully!")
print("Total Houses:", len(df))
display(df.head())

Cloning into 'Houses-dataset'...
remote: Enumerating objects: 2166, done.
remote: Counting objects: 100% (1/1), done.
remote: Total 2166 (delta 0), reused 0 (delta 0), pack-reused 2165 (from 1)
Receiving objects: 100% (2166/2166), 176.26 MiB | 38.26 MiB/s, done.
Resolving deltas: 100% (20/20), done.
Updating files: 100% (2144/2144), done.
 Dataset Downloaded Successfully!
Total Houses: 535


,Bedrooms,Bathrooms,Area,ZipCode,Price
0,4,4.0,4053,85255,869500
1,4,3.0,3343,36372,865200
2,3,4.0,3923,85266,889000
3,5,5.0,4022,85262,910000
4,3,4.0,4116,85266,971226


In [ ]:
import cv2
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

print(" Processing Images and Tabular Data... (Wait a few seconds)")

images = []
valid_indices = []
base_path = "/content/Houses-dataset/Houses Dataset/"

# 1. Load corresponding frontal images for each house
for i in df.index:
    # Dataset image names start from 1 (e.g., 1_frontal.jpg)
    img_path = f"{base_path}{i+1}_frontal.jpg"
    image = cv2.imread(img_path)

    if image is not None:
        image = cv2.resize(image, (64, 64)) # Resizing for faster training
        images.append(image)
        valid_indices.append(i)

# 2. Convert to numpy array and normalize pixel values (0 to 1)
images = np.array(images) / 255.0

# 3. Filter tabular data to match valid loaded images
df_valid = df.loc[valid_indices].copy()

# 4. Scale Tabular Data
scaler = MinMaxScaler()
tabular_features = scaler.fit_transform(df_valid[["Bedrooms", "Bathrooms", "Area", "ZipCode"]])
prices = df_valid["Price"].values

# 5. Split data into Training and Testing sets
split = train_test_split(tabular_features, images, prices, test_size=0.2, random_state=42)
(train_tab, test_tab, train_img, test_img, train_price, test_price) = split

# 6. Scale Prices for stable Neural Network training
max_price = train_price.max()
train_price = train_price / max_price
test_price = test_price / max_price

print(f"\n Data Ready!")
print(f"Total Valid Images Loaded: {len(images)}")
print(f"Tabular Data Shape: {tabular_features.shape}")

 Processing Images and Tabular Data... (Wait a few seconds)

 Data Ready!
Total Valid Images Loaded: 535
Tabular Data Shape: (535, 4)


In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, Conv2D, MaxPooling2D, Flatten, concatenate
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

print(" Building Multimodal Neural Network...")

# 1. Tabular Branch (Text Data)
tab_input = Input(shape=(4,), name="tabular_input")
x = Dense(8, activation="relu")(tab_input)
tab_branch = Dense(4, activation="relu")(x)

# 2. Image Branch (CNN for Images)
img_input = Input(shape=(64, 64, 3), name="image_input")
y = Conv2D(16, (3, 3), activation="relu")(img_input)
y = MaxPooling2D(pool_size=(2, 2))(y)
y = Conv2D(32, (3, 3), activation="relu")(y)
y = MaxPooling2D(pool_size=(2, 2))(y)
y = Flatten()(y)
img_branch = Dense(16, activation="relu")(y)

# 3. Feature Fusion (combine both!)
combined = concatenate([tab_branch, img_branch])
z = Dense(16, activation="relu")(combined)
output = Dense(1, activation="linear", name="price_output")(z)

# 4. Compile Model
model = Model(inputs=[tab_input, img_input], outputs=output)
model.compile(optimizer="adam", loss="mean_absolute_error")

# 5. Train the Model
print(" Training started...")
history = model.fit(
    x=[train_tab, train_img], y=train_price,
    validation_data=([test_tab, test_img], test_price),
    epochs=50, batch_size=8, verbose=0 # verbose=0 to avoid screen from long text
)

# 6. Evaluate Performance (MAE & RMSE)
print("\n Calculating MAE and RMSE...")
preds = model.predict([test_tab, test_img])

# Rescale prices back to original dollars
preds_real = preds.flatten() * max_price
test_price_real = test_price * max_price

mae = mean_absolute_error(test_price_real, preds_real)
rmse = np.sqrt(mean_squared_error(test_price_real, preds_real))

print(f"\n Task 3 Successfully Completed!")
print(f" MAE (Mean Absolute Error): ${mae:,.2f}")
print(f" RMSE (Root Mean Squared Error): ${rmse:,.2f}")

 Building Multimodal Neural Network...
 Training started...

 Calculating MAE and RMSE...
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step

 Task 3 Successfully Completed!
 MAE (Mean Absolute Error): $240,182.52
 RMSE (Root Mean Squared Error): $325,031.79


In [ ]:
import gradio as gr
import cv2
import numpy as np

# 1. Prediction Function
def predict_house_price(img, bedrooms, bathrooms, area, zipcode):
    if img is None:
        return "Please upload an image of the house!"

    # Image process kryn (64x64 aur normalize)
    img_resized = cv2.resize(img, (64, 64))
    img_array = np.array([img_resized]) / 255.0

    # Tabular data process kryn (scaler purane cell se aayega)
    tab_data = np.array([[bedrooms, bathrooms, area, zipcode]])
    tab_scaled = scaler.transform(tab_data)

    # Model se prediction!
    pred = model.predict([tab_scaled, img_array], verbose=0)
    pred_real = pred[0][0] * max_price

    return f"Estimated House Price: ${pred_real:,.2f}"

# 2. Pyara sa UI Setup
interface = gr.Interface(
    fn=predict_house_price,
    inputs=[
        gr.Image(label="House Image"),
        gr.Number(label="Bedrooms", value=3),
        gr.Number(label="Bathrooms", value=2),
        gr.Number(label="Area (sq ft)", value=1500),
        gr.Number(label="ZipCode", value=85255)
    ],
    outputs=gr.Text(label="AI Prediction Result"),
    title=" Multimodal House Price Predictor",
    description="Ghar ki tasveer upload kryn aur details dain, AI price batayega!"
)

# 3. Launch!
interface.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://0a34df9ef1b1f2af94.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
